### **BóSight: Data Exploration**

#### Verifies the MmCows dataset structure, conventions, and sensor synchronisation.

#### Performs six checks:
####    1. Image-label 1:1 pairing in cam_1 for the annotated day (25 July 2023).
####    2. Bounding-box parsing and per-frame visualisation with cow IDs.
####    3. Cow appearance distribution across cam_1.
####    4. Cross-modality timestamp alignment between the image stream and IMU.
####    5. Behaviour vocabulary discovery from the per-cow behaviour CSVs.
####    6. Behaviour-code verification via IMU acceleration statistics.
#
#### Matplotlib figures are saved to OUTPUT_DIR (defined below).
#### **Note:** paths below assume a Colab environment with Google Drive mounted
#### at /content/drive. Update BASE if running elsewhere.

In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; assuming /content/drive is already mounted.")

In [ ]:
# Imports
import os
from collections import Counter
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

In [ ]:
# Paths and constants
BASE   = "/content/drive/MyDrive/MmCows/extracted"
VISUAL = f"{BASE}/visual_data"
SENSOR = f"{BASE}/sensor_data"

# Single camera on the one manually annotated day.
IMG_DIR      = f"{VISUAL}/images/0725/cam_1"
LABEL_DIR    = f"{VISUAL}/labels/combined/0725/cam_1"
BEHAVIOR_DIR = f"{VISUAL}/behavior_labels/individual"

# All MmCows timestamps are Unix epoch seconds in Central Daylight Time
# (UTC - 5). Not UTC. Not local.
CDT_OFFSET = timedelta(hours=-5)

# MmCows behaviour code -> human-readable label. The paper defines seven
# behaviours; code 0 is an additional "not visible / unknown" bucket.
BEHAVIOR_CODE_TO_NAME = {
    0: "unknown",
    1: "walking",
    2: "standing",
    3: "feeding_head_up",
    4: "feeding_head_down",
    5: "licking",
    6: "drinking",
    7: "lying",
}

# BóSight uses four coarser classes. Rare or ambiguous codes are excluded
# from training data.
BEHAVIOR_CODE_TO_BOSIGHT = {
    0: None,        # exclude: cow not visible
    1: "moving",
    2: "standing",
    3: "feeding",
    4: "feeding",
    5: None,        # exclude: only 0.6% of cow-seconds
    6: None,        # exclude: only 1.0% of cow-seconds
    7: "lying",
}

OUTPUT_DIR = "/content/exploration_figs"

In [ ]:
# Utility: timestamp conversion
def unix_ts_to_cdt(unix_ts: float) -> datetime:
    """Convert a MmCows Unix timestamp to a timezone-aware CDT datetime."""
    return datetime.fromtimestamp(unix_ts, tz=timezone.utc) + CDT_OFFSET


In [ ]:
# Utility: label file parsing
def load_frame_labels(frame_stem: str) -> pd.DataFrame:
    """
    Parse a single MmCows label file.

    Each line has five whitespace-separated fields:
        cow_id  x_center  y_center  width  height
    where the last four values are normalised (0..1) to image dimensions.
    The first column is the cow ID (1..16), not a YOLO class index.

    Returns a DataFrame with one row per bounding box.
    """
    label_path = os.path.join(LABEL_DIR, f"{frame_stem}.txt")
    with open(label_path) as f:
        lines = [line for line in f.read().strip().split("\n") if line]

    rows = []
    for line in lines:
        parts = line.split()
        rows.append({
            "cow_id":        int(parts[0]),
            "x_center_norm": float(parts[1]),
            "y_center_norm": float(parts[2]),
            "w_norm":        float(parts[3]),
            "h_norm":        float(parts[4]),
        })
    return pd.DataFrame(rows)

In [ ]:
# Reference values used throughout the checks below
reference_frame = "1690271846_02-57-26"
reference_cow = "C02"
reference_ts = 1690271846

#### 1. Image-Label Pairing

In [ ]:
def check_image_label_pairing() -> None:
    """Confirm every cam_1 image has a matching label and vice versa."""
    print("\n[1] Image <-> label pairing")

    img_files = os.listdir(IMG_DIR)
    lbl_files = os.listdir(LABEL_DIR)

    img_stems = {f.replace(".jpg", "") for f in img_files}
    lbl_stems = {f.replace(".txt", "") for f in lbl_files}

    print(f"    cam_1 images:        {len(img_files)}")
    print(f"    cam_1 labels:        {len(lbl_files)}")
    print(f"    matched pairs:       {len(img_stems & lbl_stems)}")
    print(f"    images without label:{len(img_stems - lbl_stems)}")
    print(f"    labels without image:{len(lbl_stems - img_stems)}")

In [ ]:
check_image_label_pairing()

#### 2. Frame Visualisation

In [ ]:
def visualize_frame(frame_stem: str, save_as: str = None) -> None:
    """
    Load one cam_1 frame and its labels, draw bboxes with cow ID annotations.

    Used to visually confirm that bboxes land on cows, that the coordinate
    system is normalised (0..1), and that IDs match the C01-C16 convention.
    """
    print(f"\n[2] Visualising frame {frame_stem}")

    img = Image.open(os.path.join(IMG_DIR, f"{frame_stem}.jpg"))
    W, H = img.size
    print(f"    image size: {W}x{H}")

    labels = load_frame_labels(frame_stem)
    print(f"    cows in this frame: {len(labels)} "
          f"(IDs: {sorted(labels['cow_id'].tolist())})")

    fig, ax = plt.subplots(figsize=(16, 10))
    ax.imshow(img)
    cmap = plt.cm.tab20

    for _, row in labels.iterrows():
        # Denormalise from 0..1 to pixel coordinates.
        cow_id = int(row["cow_id"])
        x_c = row["x_center_norm"] * W
        y_c = row["y_center_norm"] * H
        w   = row["w_norm"] * W
        h   = row["h_norm"] * H

        # matplotlib Rectangle takes the top-left corner, not the centre.
        x1 = x_c - w / 2
        y1 = y_c - h / 2
        color = cmap(cow_id % 20)

        ax.add_patch(patches.Rectangle(
            (x1, y1), w, h,
            linewidth=3, edgecolor=color, facecolor="none",
        ))
        ax.text(
            x1, y1 - 10, f"C{cow_id:02d}",
            color="white", fontsize=14, fontweight="bold",
            bbox=dict(facecolor=color, edgecolor="none", alpha=0.9),
        )

    unix_ts = int(frame_stem.split("_")[0])
    cdt = unix_ts_to_cdt(unix_ts)
    ax.set_title(
        f"Frame {frame_stem}\n{cdt.strftime('%Y-%m-%d %H:%M:%S CDT')}",
        fontsize=14,
    )
    ax.axis("off")

    if save_as:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        out_path = os.path.join(OUTPUT_DIR, save_as)
        plt.tight_layout()
        plt.savefig(out_path, dpi=100, bbox_inches="tight")
        print(f"    saved to {out_path}")
    plt.close(fig)

In [ ]:
visualize_frame(reference_frame, save_as="reference_frame.png")

#### 3. Cow Appearance Distribution

In [ ]:
def cow_appearance_stats() -> Counter:
    """
    Count how many frames each cow appears in. Highlights class imbalance
    in the identification training data and identifies empty frames that
    can be used as detection negatives.
    """
    print("\n[3] Cow appearances across cam_1")

    counts = Counter()
    frames_with_cows = 0

    for lbl_file in os.listdir(LABEL_DIR):
        with open(os.path.join(LABEL_DIR, lbl_file)) as f:
            lines = f.read().strip().split("\n")

        if not lines or not lines[0]:
            continue
        frames_with_cows += 1

        for line in lines:
            counts[int(line.split()[0])] += 1

    total_frames = len(os.listdir(LABEL_DIR))
    print(f"    frames with >=1 cow: {frames_with_cows} / {total_frames}")
    print(f"    empty frames:        {total_frames - frames_with_cows}")

    for cow_id in sorted(counts.keys()):
        pct = 100 * counts[cow_id] / frames_with_cows
        print(f"    C{cow_id:02d}: {counts[cow_id]:5d} frames ({pct:5.1f}%)")

    return counts


In [ ]:
cow_appearance_stats()

#### 4. Cross-Modality Timestamp Alignment

In [ ]:
def check_imu_alignment(cow_id: str, target_ts: int) -> None:
    """
    Look up IMU readings for one cow at a target frame timestamp.

    Verifies:
      - IMU file exists for the C -> T mapping (C0n <-> T0n for n=1..10).
      - Timestamps use the same Unix CDT epoch as image filenames.
      - IMU sampling rate is 10 Hz as documented.

    A stationary cow shows |a| ~ 9.81 m/s^2 (gravity only) with low std.
    A walking cow shows |a| well above gravity with high std.
    """
    print(f"\n[4] IMU alignment for {cow_id} at ts={target_ts}")
    print(f"    -> {unix_ts_to_cdt(target_ts).strftime('%Y-%m-%d %H:%M:%S CDT')}")

    # Map C-prefix (visual) to T-prefix (sensor folder).
    tag_id = cow_id.replace("C", "T")
    imu_path = f"{SENSOR}/main_data/immu/{tag_id}/{tag_id}_0725.csv"

    if not os.path.exists(imu_path):
        print(f"    file not found: {imu_path}")
        return

    imu = pd.read_csv(imu_path)
    print(f"    IMU rows for the day: {len(imu)} (expected 864,000 at 10 Hz)")

    # A +/- 0.5 s window at 10 Hz gives ~10 samples.
    window = imu[(imu["timestamp"] >= target_ts - 0.5) &
                 (imu["timestamp"] <= target_ts + 0.5)]

    if len(window) == 0:
        print("    no IMU samples in the +/- 0.5 s window")
        return

    a_mag = np.sqrt(window["accel_x_mps2"]**2 +
                    window["accel_y_mps2"]**2 +
                    window["accel_z_mps2"]**2)
    print(f"    samples in +/- 0.5 s: {len(window)}")
    print(f"    |a| mean: {a_mag.mean():.2f} m/s^2  (9.81 ~ stationary)")
    print(f"    |a| std:  {a_mag.std():.3f}       (< 0.3 stationary; > 1.5 walking)")

In [ ]:
check_imu_alignment(cow_id=reference_cow, target_ts=reference_ts)

#### 5. Behaviour Vocabulary

In [ ]:
def discover_behavior_vocabulary() -> Counter:
    """
    Enumerate all behaviour codes across all 16 cows and report the
    frequency distribution.
    """
    print("\n[5] Behaviour vocabulary")

    counts = Counter()
    for cow_num in range(1, 17):
        path = f"{BEHAVIOR_DIR}/C{cow_num:02d}_0725.csv"
        df = pd.read_csv(path)
        for code in df["behavior"]:
            counts[code] += 1

    total = sum(counts.values())
    print(f"    codes observed: {sorted(counts.keys())}")
    print(f"    total cow-seconds: {total} "
          f"(expected 16 x 86400 = {16*86400})\n")

    for code in sorted(counts.keys()):
        pct = 100 * counts[code] / total
        label = BEHAVIOR_CODE_TO_NAME.get(code, "?")
        target = BEHAVIOR_CODE_TO_BOSIGHT.get(code) or "EXCLUDE"
        print(f"    code {code} ({label:<18}) {counts[code]:8d} "
              f"({pct:5.1f}%)  ->  {target}")

    return counts


In [ ]:
discover_behavior_vocabulary()

#### 6. Verify the behaviour-code mapping via IMU statistics

In [ ]:
def verify_behavior_mapping_via_imu(cow_id: str = "C02") -> None:
    """
    For each behaviour code, sample one representative timestamp and check
    the IMU acceleration statistics. Expected pattern:

        walking            -> highest std   (locomotion)
        feeding head up    -> high std      (head reaching)
        feeding head down  -> moderate std  (chewing)
        standing / lying   -> very low std  (stationary; distinguished by
                                             posture, not motion)
    """
    print(f"\n[6] Behaviour-code IMU verification (cow {cow_id})")

    tag_id = cow_id.replace("C", "T")
    imu = pd.read_csv(f"{SENSOR}/main_data/immu/{tag_id}/{tag_id}_0725.csv")
    beh = pd.read_csv(f"{BEHAVIOR_DIR}/{cow_id}_0725.csv")

    print(f"    {'code':<6}{'label':<20}{'n_sec':<10}"
          f"{'|a|_mean':<12}{'|a|_std':<10}")
    print("    " + "-" * 60)

    for code in sorted(beh["behavior"].unique()):
        rows = beh[beh["behavior"] == code]
        n_sec = len(rows)

        # Use the median-position timestamp for this code to avoid the very
        # start / end of the day.
        sample_ts = int(rows.iloc[n_sec // 2]["timestamp"])
        window = imu[(imu["timestamp"] >= sample_ts - 0.5) &
                     (imu["timestamp"] <= sample_ts + 0.5)]

        if len(window) == 0:
            print(f"    code {code}: no IMU data at sampled ts")
            continue

        a_mag = np.sqrt(window["accel_x_mps2"]**2 +
                        window["accel_y_mps2"]**2 +
                        window["accel_z_mps2"]**2)
        label = BEHAVIOR_CODE_TO_NAME.get(code, "?")
        print(f"    {code:<6}{label:<20}{n_sec:<10}"
              f"{a_mag.mean():<12.3f}{a_mag.std():<10.3f}")

In [ ]:
verify_behavior_mapping_via_imu(cow_id=reference_cow)